# LogAI - Log Preprocessing

## Objective

This notebook reads raw telecom logs, parses them into structured format,
extracts important features, performs preprocessing, and saves the processed
dataset for downstream machine learning models.

### Input
- Raw Log File (.log)

### Output
- processed_logs.csv
- parsed_logs.parquet
- feature_summary.csv

In [1]:

import pandas as pd
import numpy as np
import re
import warnings
from pathlib import Path
from tqdm import tqdm

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

Defining Project Paths

In [2]:
# Project Root

PROJECT_ROOT = Path("..")

DATA_DIR = PROJECT_ROOT / "data"

RAW_LOG_DIR = DATA_DIR / "raw_logs"

PROCESSED_DIR = DATA_DIR / "processed"

EMBEDDING_DIR = DATA_DIR / "embeddings"

print(PROJECT_ROOT)

..


Reading log files

In [3]:
LOG_FILE = RAW_LOG_DIR / "SAL_SMSC_APP1_SMPPServer_81_16_05_2026_02_46_17_7723.log"

with open(LOG_FILE, "r", encoding="utf-8", errors="ignore") as file:
    logs = file.readlines()

print("=" * 50)
print("Total Logs :", len(logs))
print("=" * 50)

Total Logs : 459617


Quick Log Inspection

In [4]:
for i in range(10):
    print(logs[i])

[ESME.cpp|01321|02:46:17:014|~CR~][CBS_Production.6940066][0x3f86f58d]Positive SMSC Resp:2010283177

[ESME.cpp|00448|02:46:17:059|~CR~][SimulQ2.8240000][0x0002ca92][SPkTh No.:0][Count:1]Inserted SMPP Packet into Queue

[Globals.cpp|00107|02:46:17:059|~CR~]Checkign for username:SimulQ2,OA:Yas

[Globals.cpp|00124|02:46:17:060|~CR~]Checkign for Account:824,DA:255653530543

[ESME.cpp|04785|02:46:17:060|~CR~]A2P Authentication Success

[Globals.cpp|04087|02:46:17:060|~CR~]AccountId:824 TPS:1

[Globals.cpp|04124|02:46:17:060|~ER~]Received TPS in this second:8

[Globals.cpp|04136|02:46:17:060|~CR~][77d27caf]Sent Byte to Router:smi: 2010283183 teleservice_id: 4098 originating_address: "Yas" destination_address: "255653530543" message_id: 135 message_type: SMS_DELIVER msg_encode: 0 user_data_type: 1 user_data_length: 42 user_data: "\005\000\003\312\002\0020 au zaidi. Asante kwa kuchagua Yas!" user_response_code: 255 time: "2026/05/16 02:46:17" validity_period: 1778975177 delivery_time: "2026/05

Empty Record List

In [7]:
records = []

Log Parser Function

In [10]:
import re

def parse_log(log):

    record = {}

    log = log.strip()

    record["Raw_Log"] = log

    # -------------------------------------------------
    # Basic Metadata
    # -------------------------------------------------

    component = re.search(r'\[(.*?)\|', log)
    line_number = re.search(r'\|(\d{5})\|', log)
    timestamp = re.search(r'(\d{2}:\d{2}:\d{2}:\d{3})', log)
    severity = re.search(r'(~CR~|~ER~|~WR~)', log)

    record["Component"] = component.group(1) if component else None
    record["Line_Number"] = line_number.group(1) if line_number else None
    record["Timestamp"] = timestamp.group(1) if timestamp else None
    record["Severity"] = severity.group(1) if severity else None

    # -------------------------------------------------
    # User Information
    # -------------------------------------------------

    username = re.search(r'username:([^,\]]+)', log)

    account = re.search(r'AccountId:(\d+)', log)

    oa = re.search(r'OA:([^,\]]+)', log)

    da = re.search(r'DA:(\d+)', log)

    record["Username"] = username.group(1) if username else None
    record["AccountID"] = account.group(1) if account else None
    record["Originating_Address"] = oa.group(1) if oa else None
    record["Destination_Address"] = da.group(1) if da else None

    # -------------------------------------------------
    # Performance Metrics
    # -------------------------------------------------

    tps = re.search(r'TPS:\s*(\d+)', log)

    queue = re.search(r'Count:(\d+)', log)

    packet = re.search(r'SPKTh No.:(\d+)', log)

    record["TPS"] = tps.group(1) if tps else None
    record["Queue_Count"] = queue.group(1) if queue else None
    record["Packet_Number"] = packet.group(1) if packet else None

    # -------------------------------------------------
    # Boolean Flags
    # -------------------------------------------------

    record["AuthenticationSuccess"] = int(
        "Authentication Success" in log
    )

    record["PositiveResponse"] = int(
        "Positive SMSC Resp" in log
    )

    record["QueueInserted"] = int(
        "Inserted SMPP Packet" in log
    )

    record["RouterSent"] = int(
        "Sent Byte to Router" in log
    )

    record["ContainsError"] = int(
        any(x in log.upper() for x in [
            "ERROR",
            "FAILED",
            "TIMEOUT",
            "NOT FOUND"
        ])
    )

    # -------------------------------------------------
    # Text Features
    # -------------------------------------------------

    record["Message_Length"] = len(log)

    record["Word_Count"] = len(log.split())

    return record

In [11]:
records = []

for log in tqdm(logs):
    records.append(parse_log(log))

df = pd.DataFrame(records)

df.head()

100%|██████████| 459617/459617 [00:03<00:00, 128518.72it/s]


,Raw_Log,Component,Line_Number,Timestamp,Severity,Username,AccountID,Originating_Address,Destination_Address,TPS,Queue_Count,Packet_Number,AuthenticationSuccess,PositiveResponse,QueueInserted,RouterSent,ContainsError,Message_Length,Word_Count
0,[ESME.cpp|01321|02:46:17:014|~CR~][CBS_Production.6940066][0x3f86f58d]Positive SMSC Resp:2010283177,ESME.cpp,01321,02:46:17:014,~CR~,NaN,NaN,NaN,NaN,NaN,NaN,None,0,1,0,0,0,99,3
1,[ESME.cpp|00448|02:46:17:059|~CR~][SimulQ2.8240000][0x0002ca92][SPkTh No.:0][Count:1]Inserted SMPP Packet into Queue,ESME.cpp,00448,02:46:17:059,~CR~,NaN,NaN,NaN,NaN,NaN,1,None,0,0,1,0,0,116,6
2,"[Globals.cpp|00107|02:46:17:059|~CR~]Checkign for username:SimulQ2,OA:Yas",Globals.cpp,00107,02:46:17:059,~CR~,SimulQ2,NaN,Yas,NaN,NaN,NaN,None,0,0,0,0,0,73,3
3,"[Globals.cpp|00124|02:46:17:060|~CR~]Checkign for Account:824,DA:255653530543",Globals.cpp,00124,02:46:17:060,~CR~,NaN,NaN,NaN,255653530543,NaN,NaN,None,0,0,0,0,0,77,3
4,[ESME.cpp|04785|02:46:17:060|~CR~]A2P Authentication Success,ESME.cpp,04785,02:46:17:060,~CR~,NaN,NaN,NaN,NaN,NaN,NaN,None,1,0,0,0,0,60,3


In [12]:
df["Timestamp"] = pd.to_datetime(
    df["Timestamp"],
    format="%H:%M:%S:%f",
    errors="coerce"
)

numeric_cols = [
    "Line_Number",
    "AccountID",
    "TPS",
    "Queue_Count",
    "Packet_Number"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [13]:
df["Hour"] = df["Timestamp"].dt.hour
df["Minute"] = df["Timestamp"].dt.minute
df["Second"] = df["Timestamp"].dt.second
df["Millisecond"] = (
    df["Timestamp"].dt.microsecond // 1000
)

In [14]:
for col in df.columns:

    if df[col].dtype == "object":

        df[col] = df[col].fillna("Unknown")

    else:

        df[col] = df[col].fillna(0)

In [15]:
print("Before :", len(df))

df = df.drop_duplicates()

print("After :", len(df))

Before : 459617
After : 455759


In [17]:
summary = pd.DataFrame({

    "Missing Values": df.isnull().sum(),

    "Unique Values": df.nunique(),

    "Data Type": df.dtypes.astype(str)

})

summary

,Missing Values,Unique Values,Data Type
Raw_Log,0,455759,str
Component,0,6,str
Line_Number,0,58,int64
Timestamp,0,164576,datetime64[us]
Severity,0,2,str
Username,0,61,object
AccountID,0,67,float64
Originating_Address,0,2818,object
Destination_Address,0,31989,object
TPS,0,25,float64


In [21]:
# Convert all object columns to string

object_cols = df.select_dtypes(include=["object"]).columns

for col in object_cols:
    df[col] = df[col].astype(str)

print("Converted object columns to string.")

Converted object columns to string.


In [22]:
df.to_parquet(
    processed_path / "parsed_logs.parquet",
    index=False
)

In [23]:
import os

os.listdir("../data/processed")

['parsed_logs.parquet', 'processed_logs.csv']

In [24]:
from pathlib import Path

for file in Path("../data/processed").iterdir():
    print(file.name)

parsed_logs.parquet
processed_logs.csv


In [25]:
from pathlib import Path

processed_path = Path("../data/processed")

processed_path.mkdir(parents=True, exist_ok=True)

df.to_csv(
    processed_path / "processed_logs.csv",
    index=False
)

df.to_parquet(
    processed_path / "parsed_logs.parquet",
    index=False
)

summary.to_csv(
    processed_path / "feature_summary.csv"
)

print("✅ Files Saved Successfully")

✅ Files Saved Successfully
